In [ ]:
from analysis_helper import *

import country_converter as coco

cc = coco.CountryConverter()

focus_carrier = [
    'nuclear',         # Baseload, always on
    'lignite',         # Baseload, slow to ramp
    'coal',            # Dispatchable, used as baseload or mid-merit
    'oil',             # Peaker, dispatchable, expensive
    'CCGT',            # Flexible, mid-merit
    'OCGT',            # Peaker, fast-ramping
    'biomass',         # Dispatchable, renewable, relatively stable
    'biomass EOP',         # Dispatchable, renewable, relatively stable
    'geothermal',      # Very stable, constant output
    'hydro',           # Controllable to some extent, seasonal
    'ror',             # Run-of-river, weather-dependent
    'offwind-dc',      # Offshore wind, variable but often more stable
    'offwind-ac',      # Same as above, different transmission
    'onwind',          # Land-based wind, variable
    'solar',           # Daylight only, weather-sensitive
    'solar rooftop'    # Most variable, decentralized, small-scale
]

savefig = False

# Generate Regional Energy Balance

In [ ]:
def calculate_energy_balance(n):
    available_countries = n.buses.country.unique()
    filtered_countries = [c for c in country_prefered_order if c in available_countries]
    
    # ---- Energy Balance (Generation) ----
    df_gen = pd.DataFrame(get_energy_balance_bus(n)).reset_index()
    df_gen = df_gen[df_gen["carrier"].isin(focus_carrier)]
    df_gen["country"] = df_gen["bus"].map(n.buses.country)
    df_gen = df_gen.groupby(["carrier", "country"]).sum()[0].unstack("carrier").fillna(0)
    df_gen = df_gen.reindex(filtered_countries)
    df_gen.index = [coco.convert(names=name, to='short') if name in set(cc.data['ISO2']) else name for name in df_gen.index]
    df_gen /= 1e6  # Convert MWh to TWh
    
    # ---- Energy Balance (Load) ----
    df_load = pd.DataFrame(get_energy_balance_load(n)).reset_index()
    df_load["country"] = df_load["bus"].map(n.buses.country)
    df_load = df_load.groupby("country").sum()[[0]]
    df_load = df_load.reindex(filtered_countries)
    df_load.index = [coco.convert(names=name, to='short') if name in set(cc.data['ISO2']) else name for name in df_load.index]
    df_load /= 1e6  # Convert MWh to TWh
    df_load.columns = ["Load"]

    return df_gen, df_load


def plot_energy_balance(n, ax, title):

    df_gen, df_load = calculate_energy_balance(n)
    
    # Plot generation (stacked bars)
    df_gen.plot(
        kind="bar",
        stacked=True,
        color=[n.carriers.color[c] for c in df_gen.columns],
        width=0.9,
        ax=ax,
        legend=False
    )
    
    # Overlay load (outlined bars)
    bottom = [0] * len(df_load)
    for col in df_load.columns:
        values = df_load[col].values
        ax.bar(
            range(len(df_load)),
            values,
            bottom=bottom,
            width=0.9,
            facecolor='none',
            edgecolor='black',
            linestyle='--', 
            linewidth=1.5,
            label=col
        )
        bottom = [b + v for b, v in zip(bottom, values)]
    
    ax.set_title(title)
    ax.set_ylabel("Energy [TWh]")
    ax.set_ylim([0,700])
    ax.grid(True, axis='y', linestyle='--', alpha=0.7)

    return ax


def plot_energy_balance_compare(n_before, n_after, ax, title):

    df_before, _ = calculate_energy_balance(n_before)
    df_after, df_load = calculate_energy_balance(n_after)
    
    df_gen = ((df_after - df_before).T/df_load["Load"]*100).T
    
    # Plot generation (stacked bars)
    df_gen.plot(
        kind="bar",
        stacked=True,
        color=[n.carriers.color[c] for c in df_gen.columns],
        width=0.9,
        ax=ax,
        legend=False
    )
    
    ax.set_title(title)
    ax.set_ylabel("Change relative to load [%]")
    ax.set_ylim([-100,100])
    ax.grid(True, axis='y', linestyle='--', alpha=0.7)

    return ax

In [ ]:
year = 2050

scenario = "decarbonize-existing-3H"
n0 = pypsa.Network(f"../results/{scenario}/postnetworks/elec_s_100_ec_lv2.0__3h_{year}_0.071_DEC_0export.nc")
n0.buses["subregion"] = n0.buses.index.map(assign_region_or_country)
n0.buses.country = n0.buses["subregion"].fillna(n0.buses.country)

scenario = "decarbonize-aims-3H"
n = pypsa.Network(f"../results/{scenario}/postnetworks/elec_s_100_ec_lv2.0__3h_{year}_0.071_DEC_0export.nc")
n.buses["subregion"] = n.buses.index.map(assign_region_or_country)
n.buses.country = n.buses["subregion"].fillna(n.buses.country)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

plot_energy_balance(n, ax1, f"{scenario} - Year {year}")
plot_energy_balance_compare(n0, n, ax2, f"Comparing existing with aims")

handles, labels = ax1.get_legend_handles_labels()
labels = ['Demand' if x == 'Load' else x for x in labels]
handles, labels = handles[::-1], labels[::-1]

nice_name = n.carriers.nice_name.map(nice_name_rename).fillna(n.carriers.nice_name)
labels = [nice_name[lable] if lable in nice_name else lable for lable in labels]

# --- Add ONE shared legend at bottom center ---
fig.legend(
    handles,
    labels,
    title="Carrier",
    loc="lower center",
    ncol=4,                 # adjust number of columns as needed
    bbox_to_anchor=(0.5, -0.35),   # moves legend below the figure
)

# --- Make room for legend ---
plt.subplots_adjust(bottom=0.1, hspace=0.15)   # increase space at bottom

if savefig:
    fig.savefig(
        "aims-energy-regional-3H.png",
        dpi=300,
        bbox_inches="tight",
    )

plt.show()

In [ ]:
year = 2050

scenario = "baseline-existing-3H"
n0 = pypsa.Network(f"../results/{scenario}/postnetworks/elec_s_100_ec_lv2.0__3h_{year}_0.071_DEC_0export.nc")
n0.buses["subregion"] = n0.buses.index.map(assign_region_or_country)
n0.buses.country = n0.buses["subregion"].fillna(n0.buses.country)

scenario = "baseline-aims-3H"
n = pypsa.Network(f"../results/{scenario}/postnetworks/elec_s_100_ec_lv2.0__3h_{year}_0.071_DEC_0export.nc")
n.buses["subregion"] = n.buses.index.map(assign_region_or_country)
n.buses.country = n.buses["subregion"].fillna(n.buses.country)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

plot_energy_balance(n, ax1, f"{scenario} - Year {year}")
plot_energy_balance_compare(n0, n, ax2, f"Comparing existing with aims")

handles, labels = ax2.get_legend_handles_labels()
handles, labels = handles[::-1], labels[::-1]

nice_name = n.carriers.nice_name.map(nice_name_rename).fillna(n.carriers.nice_name)
labels = [nice_name[lable] if lable in nice_name else lable for lable in labels]

# --- Add ONE shared legend at bottom center ---
fig.legend(
    handles,
    labels,
    title="Carrier",
    loc="lower center",
    ncol=4,                 # adjust number of columns as needed
    bbox_to_anchor=(0.5, -0.35),   # moves legend below the figure
)

# --- Make room for legend ---
plt.subplots_adjust(bottom=0.1, hspace=0.15)   # increase space at bottom

if savefig:
    fig.savefig(
        f"{scenario}-energy-regional-3H.png",
        dpi=300,
        bbox_inches="tight",
    )

plt.show()

# Generate Price Distribution

In [ ]:
def violin_comp(n, ax, title="", ymax=1000):
    df = n.statistics.prices(groupby_time=False, bus_carrier=["low voltage"])
    df_w = n.statistics.energy_balance(groupby=["bus"], comps=["Load"], bus_carrier=["low voltage"], groupby_time=False)
    df_w = df_w.groupby("bus").sum()

    # weighted average between nodes
    denominator = (
        df_w.groupby(df_w.index.map(n.buses.country))
            .transform("sum")
            .reindex(df.index)   # ensures alignment
    )
    
    df = df.mul(df_w).div(denominator)
    df = df.groupby(n.buses.country).sum()

    # weighted average between time-series
    df_w = df_w.groupby(n.buses.country).sum()
    data = (df.mul(df_w).T/df_w.T.mean()).T

    # get weighed average of mean and median
    mean = (data.T.mean() * df_w.T.sum() / df_w.sum().sum()).sum()
    median = (data.T.median() * df_w.T.sum() / df_w.sum().sum()).sum()
    
    available_countries = n.buses.country.unique()
    filtered_countries = [c for c in country_prefered_order if c in available_countries]
    data = data.reindex(filtered_countries)
    data = data.T
    
    data_array = [data[c].to_numpy() for c in data.columns]
    
    parts = ax.violinplot(data_array, showmeans=True, showmedians=True)
    
    ax.set_ylabel("Marginal Price (€/MWh)")
    ax.grid(axis="y")
    
    ax.set_ylim([0, ymax])

    for pc in parts['bodies']:
        pc.set_facecolor('lightgrey')
        pc.set_edgecolor('lightgrey')
        pc.set_alpha(1)

    for partname in ('cbars','cmins','cmaxes'):
        vp = parts[partname]
        vp.set_linewidth(0)
        
    vp = parts['cmeans']
    vp.set_edgecolor("black")
    vp.set_linewidth(6)

    vp = parts['cmaxes']
    vp.set_edgecolor("red")
    vp.set_linewidth(6)

    vp = parts['cmedians']
    vp.set_edgecolor("blue")
    vp.set_linewidth(6)
    
    col = 1
    for c in data.columns:
        data_max = data[c].max()
        if data_max > ymax:
            ax.annotate(str(round(data_max)),xy=(col,ymax), xytext=(0, -10), 
                        textcoords='offset points', ha='center', va='bottom', 
                        rotation=0, color = "red", size=8)
        col += 1

    ax.axhline(y=mean, color='black', linestyle='--', linewidth=0.9, label='Overall mean')
    ax.axhline(y=median, color='blue',  linestyle='--', linewidth=0.9, label='Overall median')

    # Add text above the lines
    ax.text(
        x=0.2,  # slightly to the right of last violin
        y=mean,
        s=f"{mean:.1f}",
        color='black',
        fontsize=8,
        va='bottom',  # vertical alignment above the line
        ha='left'     # horizontal alignment to the left
    )
    
    ax.text(
        x=0.2,
        y=median,
        s=f"{median:.1f}",
        color='blue',
        fontsize=8,
        va='bottom',
        ha='left'
    )
    
    ax.set_title(title)

    return ax

In [ ]:
fig, [ax1, ax3] = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

scenarios = {
    "decarbonize-existing-3H": ax1,
    #"decarbonize-asean-3H": ax2,
    "decarbonize-aims-3H": ax3,
}

for scenario, ax in scenarios.items():
    n = pypsa.Network(f"../results/{scenario}/postnetworks/elec_s_100_ec_lv2.0__3h_{year}_0.071_DEC_0export.nc")
    n.buses["subregion"] = n.buses.index.map(assign_region_or_country)
    n.buses.country = n.buses["subregion"].fillna(n.buses.country)
    
    ax = violin_comp(n, ax, title=f"{scenario} - Year {year}", ymax=200)

# --- Add ONE shared legend at bottom center ---
from matplotlib.lines import Line2D

legend_handles = [
    Line2D([0], [0], color="blue",  lw=0, marker='s', markersize=6, label="median"),
    Line2D([0], [0], color="black", lw=0, marker='s', markersize=6, label="mean"),
    Line2D([0], [0], color="red",   lw=0, marker='s', markersize=6, label="max"),
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.25),
    frameon=True,
    ncol=3
)

available_countries = n.buses.country.unique()
filtered_countries = [c for c in country_prefered_order if c in available_countries]
short_name = [coco.convert(names=name, to='short') if name in set(cc.data['ISO2']) else name for name in filtered_countries]
_ = plt.xticks(range(1,len(short_name)+1), short_name, rotation=90)

# --- Make room for legend ---
plt.subplots_adjust(bottom=0.1)   # increase space at bottom

if savefig:
    fig.savefig(
        f"marginal-price-3H.png",
        dpi=300,
        bbox_inches="tight",
    )

plt.show()